In [ ]:
import cv2
from ultralytics import YOLO

def main():
    global ROI_center_spot #mouse callback variable 
    # global counted_ids, counted_vec_types #to check the variable
    # load model
    model = YOLO('runs/detect/yolov8s_vehicle_detector/weights/best.pt')

    # Open the video file
    videopath = 'DJI_0004_HS0600D.MP4'
    cap = cv2.VideoCapture(videopath)

    if not cap.isOpened():
        print("Error: Could not open video stream.")
        return
    video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ROI_center_spot = (744, video_height // 2)
    box_width = 640
    box_height = 225

    cv2.namedWindow("YOLO Detection")
    cv2.setMouseCallback("YOLO Detection", choose_spot)
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"Video Width: {video_width}, Video Height: {video_height}")
    print(f"Video FPS: {fps:.2f}")
    # declaring variable
    prev_x = {} #ids in previous frame that hasnt been counted
    counted_ids = set() #id thats been counted after crossing the line
    counted_vec_types = [] #types of vehicles of counted ids
    count = 0
    toogle_first_frame = 0 #counting first few frame to avoid cluttering
    len_prev_counted_ids = 0
    frame_count = []
    current_frame = 0
    counted_coordinates = [] #to store the coordinates of counted vehicles
    while True:
        # Capture frame-by-frame
        ret, frame = cap.read()

        if not ret:
            print("Error: Could not read frame.")
            break

        # YOLO Prediction
        # prediction box
        

    
        top_left = (
            ROI_center_spot[0] - 30, 
            ROI_center_spot[1] - box_height // 2
            )
        bottom_right = (
            ROI_center_spot[0] + box_width // 2, 
            ROI_center_spot[1] + box_height // 2
            )
        x1, y1 = top_left
        x2, y2 = bottom_right
        # Ensure the box is within the frame boundaries
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(video_width, x2)
        y2 = min(video_height, y2)
        cv2.rectangle(frame, top_left, bottom_right, (0, 255, 0), 2)
        cv2.line(frame, [ROI_center_spot[0], y1], [ROI_center_spot[0], y2], (0, 255, 0), 2)

        roi = frame[y1:y2, x1:x2]
        results = model.track(roi, 
                              persist=True, 
                              conf=0.3, #try if 0.2 is too low #0.5 too high
                              iou=0.5,
                              max_det=300,
                              tracker= 'botsort.yaml') #other options: 'bytetrack.yaml', 'deepsort.yaml', 'strongsort.yaml', 'botsort.yaml', 'ocsort.yaml', 'sort.yaml'
        if results and results[0].boxes is not None:   
            for box in results[0].boxes:
                track_id = int(box.id[0]) if box.id is not None else None
                cls = int(box.cls[0])
                conf = box.conf[0]

                bx1, by1, bx2, by2 = map(int, box.xyxy[0])

                # shift to full frame
                bx1 += x1
                by1 += y1
                bx2 += x1
                by2 += y1

                cbx = (bx1 + bx2) // 2
                cby = (by1 + by2) // 2

                label = f"{model.names[cls]} {conf:.2f}"

                cv2.rectangle(frame, (bx1, by1), (bx2, by2), (0,0,255), 2)
                cv2.putText(frame, label,
                            (bx1, by1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.5,
                            (0,0,255),
                            1)
                zone_left = ROI_center_spot[0] - 20
                zone_right = ROI_center_spot[0] + 20
                if track_id in prev_x:
                    if zone_left <= cbx <= zone_right:
                    # if prev_x[track_id] > ROI_center_spot[0] and cbx <= ROI_center_spot[0] and track_id not in counted_ids:
                        count += 1
                        counted_ids.add(track_id)
                        if len(counted_ids) > len_prev_counted_ids:
                            counted_vec_types.append(model.names[cls])
                            frame_count.append(current_frame/fps)
                            counted_coordinates.append((cbx, cby))
                            len_prev_counted_ids = len(counted_ids)
                # print(f"counted_ids: {counted_ids}, len_prev_counted_ids: {len_prev_counted_ids}")
                # # print(f"Track ID: {track_id}, Class: {model.names[cls]}, Confidence: {conf:.2f}, Center: ({cbx}, {cby})")
                prev_x[track_id] = cbx
        print(f"ROI_center_spot_0: {ROI_center_spot[0]}, count aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa: {count}")
        if toogle_first_frame < 50:
            # print('prev_x :')
            # print(prev_x)
            toogle_first_frame += 1
            # # print("First frame processed, counting will start now.")
        display_counted_ids = list(counted_ids)[-3:] if len(counted_ids) > 3 else list(counted_ids)
        cv2.putText(frame, f"Counted IDs: {display_counted_ids}",
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,0,0),
                    2)
        display_counted_types = list(counted_vec_types)[-3:] if len(counted_vec_types) > 3 else list(counted_vec_types)
        cv2.putText(frame, f"Counted Types: {display_counted_types}",
                    (10, 70),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,0,0),
                    2)
        cv2.putText(frame, f"Total Counted Types: {len(counted_vec_types)}",
                    (10, 110),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,0,0),
                    2)

        ## prediction box
        # annotated_frame = results[0].plot()
        ## show results
        cv2.imshow('YOLO Detection', frame)


        current_frame += 1
        # Exit on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the capture and close windows
    cap.release()
    cv2.destroyAllWindows()

    return counted_ids, counted_vec_types, frame_count, counted_coordinates

def choose_spot(event, x, y, flags, param):
    global ROI_center_spot

    if event == cv2.EVENT_LBUTTONDOWN:
        ROI_center_spot = (x, y)
        print("Clicked:", ROI_center_spot)

In [ ]:
# tulis ulang manual kode
# tanpa trigger reposisi RoI box
import cv2
from ultralytics import YOLO
global ROI_center_spot #mouse callback

model = YOLO('runs/detect/yolov8s_vehicle_detector/weights/best.pt')
videopath = 'DJI_0004_HS0600D.MP4'

box_width = 640 #bounding box
box_height = 225

# variables
prev_x = {} #ids in previous frame
counted_id = set() #id thats been counted after crossing the line
counted_vec_types = [] #types of vehicles of counted_id
counted_coordinates = [] #coordinate of vehicles of counted_id
toggle_first_frame = 0 #toggle how many first frame to count to keep consistency and avoid cluttering
current_frame = 0 # which frame is this

# video variables
cap = cv2.VideoCapture(videopath)
if not cap.isOpened():
    print("Error: Could not open video stream.")
    exit()
video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# RoI box coordinates configuration
ROI_center_spot = (744, video_height // 2) #center spot RoI box, posisi terbaik dengan pertimbangan pohon
ROI_top_left = (
    ROI_center_spot[0] - 30, 
    ROI_center_spot[1] - box_height // 2
)
ROI_bottom_right = (
    ROI_center_spot[0] + box_width // 2, 
    ROI_center_spot[1] + box_height // 2
)
ROI_x1, ROI_y1 = ROI_top_left
ROI_x2, ROI_y2 = ROI_bottom_right

# konfigurasi cv2 window (diluar loop)
cv2.namedWindow("YOLO Detection")

# loop next frame
while True:
    ret, frame = cap.read()

    if not ret:
        print('ERROR:cannot read frame')
        break

    # visualisasi bounding box dan counting line pada frame
    cv2.rectangle(frame, ROI_top_left, ROI_bottom_right,(0, 255, 0), 2) #visual bounding box
    cv2.line(frame, [ROI_center_spot[0], ROI_y1], [ROI_center_spot[0], ROI_y2], (0, 255, 0), 2) #visual counting line

    # kode tracking
    roi = frame[ROI_y1:ROI_y2, ROI_x1:ROI_x2] # definisi RoI
    results = model.track(roi,
                          persist=True,
                          conf=0.3, #0.2~0.5
                          iou=0.5, #only iou 0.5 has been tested in result
                          max_det = 300, #
                          tracker='botsort.yaml')
    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            if box.id is not None:
                track_id = int(box.id[0])
                cls = int(box.cls[0])
                conf = box.conf[0]
                label = f'{model.names[cls]} {conf:.2f}'
                # print(f'track_id = {track_id}')
            else: #mempertahankan tipe data jika tidak ada objek terdeteksi
                track_id = None
                cls = None
                conf = None
                label = ''
            
            # perkiraan posisi bounding box
            bounding_x1, bounding_y1, bounding_x2, bounding_y2 = map(int, box.xyxy[0])

            # penyesuaian dengan frame awal
            bounding_x1 += ROI_x1
            bounding_y1 += ROI_y1
            bounding_x2 += ROI_x1
            bounding_y2 += ROI_y1

            center_bounding_x = (bounding_x1 + bounding_x2)//2
            center_bounding_y = (bounding_y1 + bounding_y2)//2

            # visualisasi bounding box pada cv2 window
            cv2.rectangle(frame, (bounding_x1, bounding_y1), (bounding_x2, bounding_y2), (0,0,255),2)
            cv2.putText(frame, label, (bounding_x1, bounding_y1 - 5),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,255),1)

            # counting line center spot
            counting_line_center = ROI_center_spot[0]

            # proses counting
            if track_id in prev_x:
                if center_bounding_x < counting_line_center:
                    counted_id.add(track_id)
            prev_x[track_id] = center_bounding_x

    cv2.imshow('YOLO Detection', frame)

    current_frame += 1
    # Exit on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the capture and close windows
cap.release()
cv2.destroyAllWindows()    



0: 192x256 6 motorcycles, 2 cars, 40.2ms
Speed: 27.9ms preprocess, 40.2ms inference, 0.6ms postprocess per image at shape (1, 3, 192, 256)

0: 192x256 5 motorcycles, 2 cars, 35.2ms
Speed: 1.2ms preprocess, 35.2ms inference, 0.9ms postprocess per image at shape (1, 3, 192, 256)

0: 192x256 5 motorcycles, 2 cars, 33.0ms
Speed: 0.7ms preprocess, 33.0ms inference, 0.7ms postprocess per image at shape (1, 3, 192, 256)

0: 192x256 5 motorcycles, 2 cars, 35.6ms
Speed: 0.7ms preprocess, 35.6ms inference, 0.5ms postprocess per image at shape (1, 3, 192, 256)

0: 192x256 6 motorcycles, 2 cars, 32.9ms
Speed: 1.1ms preprocess, 32.9ms inference, 0.5ms postprocess per image at shape (1, 3, 192, 256)

0: 192x256 6 motorcycles, 2 cars, 35.2ms
Speed: 0.8ms preprocess, 35.2ms inference, 0.7ms postprocess per image at shape (1, 3, 192, 256)

0: 192x256 6 motorcycles, 2 cars, 35.9ms
Speed: 0.9ms preprocess, 35.9ms inference, 0.6ms postprocess per image at shape (1, 3, 192, 256)

0: 192x256 6 motorcycles,

In [8]:
counted_id

{1, 3, 6, 15, 18, 19, 25, 31, 33, 41, 46}

In [5]:
counted_ids, counted_vec_types, frame_count, counted_coordinates = main()

Video Width: 1920, Video Height: 1080
Video FPS: 59.94

0: 192x256 6 motorcycles, 2 cars, 78.3ms
Speed: 39.3ms preprocess, 78.3ms inference, 1.0ms postprocess per image at shape (1, 3, 192, 256)
ROI_center_spot_0: 744, count aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa: 0

0: 192x256 5 motorcycles, 2 cars, 46.2ms
Speed: 1.3ms preprocess, 46.2ms inference, 0.7ms postprocess per image at shape (1, 3, 192, 256)
ROI_center_spot_0: 744, count aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa: 2

0: 192x256 5 motorcycles, 2 cars, 43.9ms
Speed: 1.5ms preprocess, 43.9ms inference, 0.6ms postprocess per image at shape (1, 3, 192, 256)
ROI_center_spot_0: 744, count aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa: 4

0: 192x256 5 motorcycles, 2 cars, 49.3ms
Speed: 0.9ms preprocess, 49.3ms inference, 0.7ms postprocess per image at shape (1, 3, 192, 256)
ROI_center_spot_0: 744, count aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa

In [9]:
len(counted_ids), len(counted_vec_types), len(frame_count), len(counted_coordinates)
# frame_count

(846, 846, 846, 846)

In [10]:
counted_coordinates

[(739, 595),
 (727, 571),
 (764, 559),
 (763, 578),
 (743, 559),
 (725, 575),
 (764, 580),
 (737, 579),
 (764, 576),
 (763, 556),
 (763, 577),
 (728, 574),
 (763, 579),
 (734, 576),
 (762, 588),
 (736, 580),
 (763, 595),
 (764, 541),
 (764, 565),
 (764, 574),
 (764, 596),
 (734, 575),
 (737, 597),
 (764, 588),
 (737, 590),
 (764, 564),
 (764, 563),
 (734, 562),
 (734, 562),
 (764, 560),
 (728, 560),
 (764, 585),
 (763, 582),
 (762, 580),
 (736, 580),
 (762, 575),
 (764, 553),
 (735, 574),
 (764, 567),
 (732, 564),
 (764, 575),
 (730, 576),
 (764, 586),
 (761, 591),
 (762, 574),
 (733, 573),
 (764, 551),
 (764, 584),
 (734, 582),
 (764, 582),
 (748, 582),
 (764, 549),
 (764, 587),
 (761, 580),
 (735, 586),
 (764, 560),
 (763, 576),
 (753, 564),
 (736, 575),
 (725, 560),
 (764, 591),
 (764, 574),
 (736, 591),
 (764, 562),
 (763, 584),
 (735, 574),
 (762, 582),
 (733, 584),
 (730, 559),
 (764, 581),
 (762, 578),
 (762, 564),
 (734, 577),
 (724, 563),
 (764, 585),
 (764, 569),
 (764, 558),

In [11]:
import pandas as pd
df = pd.DataFrame({'frame_count': frame_count,
                   'counted_vec_types': counted_vec_types,
                   'counted_ids': list(counted_ids),
                   'counted_coordinates': counted_coordinates})
df.to_excel('counted_data.xlsx', index=False)